[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/cours/seance4_cours.ipynb)

# Séance 2.4 — Visualiser et conclure — étude de cas

**Cours** · durée : 2h — 1h de technique en alternance, 1h d'étude de cas en binôme

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- choisir le bon graphique selon la question posée
- produire une courbe, un histogramme, un diagramme en barres et un nuage de points
- rendre un graphique lisible : titre, axes, unités
- repérer ce qu'un graphique cache autant que ce qu'il montre
- conclure une analyse par des recommandations chiffrées

## Pourquoi faire un graphique

Voici le chiffre d'affaires mensuel, sous forme de tableau :

| mois | CA | mois | CA |
|---|---|---|---|
| 2010-12 | 57 705 | 2011-06 | 75 590 |
| 2011-01 | 78 452 | 2011-07 | 107 164 |
| 2011-02 | 52 115 | 2011-08 | 85 469 |
| 2011-03 | 80 451 | 2011-09 | 133 236 |
| 2011-04 | 60 493 | 2011-10 | 170 010 |
| 2011-05 | 80 393 | 2011-11 | 133 937 |

Vous l'avez lu. Avez-vous **vu** quelque chose ?

Maintenant regardez la même chose en courbe (dans deux cellules). La tendance
saute aux yeux en une seconde.

> **Un graphique ne décore pas un rapport : il fait voir ce qu'un tableau
> cache.** Corollaire souvent oublié — si un tableau de trois lignes suffit,
> ne faites pas de graphique.

## Choisir le bon graphique

| Votre question | Le graphique |
|---|---|
| Comment ça évolue **dans le temps** ? | une **courbe** |
| Qui est le plus gros ? Comment ça se **compare** ? | des **barres** |
| Comment les valeurs sont-elles **réparties** ? | un **histogramme** |
| Y a-t-il un **lien** entre deux grandeurs ? | un **nuage de points** |

Quatre questions, quatre graphiques. C'est presque tout ce dont vous aurez
besoin.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]     ## le CA de chaque ligne
ventes["date"] = pd.to_datetime(ventes["date"])   ## du texte vers des dates

# Deux jointures enchainees : ventes + clients, puis + produits
complet = ventes.merge(clients, on="client_id").merge(produits, on="prod_id")
print(complet.shape)   ## 45 123 lignes : aucune perdue en chemin

## 1. La courbe — l'évolution dans le temps

In [ ]:
# to_period("M") regroupe toutes les dates d'un meme mois
ca_mois = ventes.groupby(ventes["date"].dt.to_period("M"))["ca"].sum()
ca_mois.index = ca_mois.index.astype(str)   ## en texte : matplotlib prefere

ca_mois.plot(kind="line", marker="o", figsize=(7, 4))   ## une evolution
plt.title("Chiffre d'affaires mensuel")   ## sans titre, ce n'est pas un livrable
plt.ylabel("CA (euros)")                  ## la grandeur ET son unite
plt.xticks(rotation=45)                   ## des dates inclinees se lisent
plt.show()

Une montée régulière jusqu'à un pic en octobre, puis une chute brutale en
décembre.

**Que concluez-vous ?** Prenez trente secondes avant de continuer.

In [ ]:
# Verifions quelque chose avant de conclure
decembre = ventes.query("date >= '2011-12-01'")

print("derniere date du fichier :", ventes["date"].max().date())   ## le 9 !
print("jours de decembre 2011 presents :", decembre["date"].dt.day.nunique())

> ⚠️ **Il n'y a pas eu d'effondrement en décembre.** Le fichier s'arrête au
> **9 décembre**. On compare 8 jours de vente à des mois complets de 30 jours.
>
> Le graphique ne mentait pas. C'est la lecture qui était fausse.

C'est **l'erreur d'analyse la plus fréquente en entreprise**, et l'une des
plus coûteuses. Avant d'interpréter une évolution, vérifiez toujours que
**toutes les périodes sont comparables**.

Le vrai pic, lui, est bien réel : octobre. Pour un grossiste, c'est logique —
les détaillants se réapprovisionnent **avant** Noël, pas pendant.

---

### ✏️ À vous 1a — La courbe, avec titre et unité

> **Votre mission :**
> - Tracer le **nombre de commandes distinctes** par mois sous forme de courbe.
> - Titre et unité obligatoires : un graphique sans légende n'est pas un graphique, c'est un dessin.
> - Incliner les étiquettes à 45° et appeler `tight_layout()` : sur un petit écran, sans ça les dates se chevauchent ou sont coupées.
> - Mettre le mois qui compte le plus de commandes dans `mois_cmd`, et le nombre de mois du fichier dans `nb_mois`.

In [ ]:
nb_cmd = complet.groupby(complet["date"].dt.to_period("M"))["cmd_id"].____()
nb_cmd.index = nb_cmd.index.astype(str)

nb_cmd.plot(kind="____", marker="o", figsize=(7, 4))
plt.title("Nombre de commandes par mois")
plt.ylabel("____")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

mois_cmd = nb_cmd.idxmax()
nb_mois = len(nb_cmd)
print(mois_cmd, "|", nb_mois, "mois")

In [ ]:
verifier("1a - mois record en commandes", mois_cmd == "2011-11",
         "nunique() compte les commandes distinctes, count() compterait les lignes")
verifier("1b - nombre de mois", nb_mois == 13,
         "decembre 2010 et decembre 2011 comptent tous les deux")

---

### ✏️ À vous 1b — Deux marchés sur la même figure

> **Votre mission :**
> - Comparer l'évolution mensuelle de la France et de l'Allemagne **sur un seul graphique**.
> - Mettre le meilleur mois français dans `mois_fr`.
> - Deux courbes sur une figure se comparent ; deux figures côte à côte, non.

In [ ]:
deux = complet.query("pays in ['France', '____']")
par_mois = deux.groupby([deux["date"].dt.to_period("M"), "pays"])["ca"].sum().unstack()
par_mois.index = par_mois.index.astype(str)

par_mois.plot(kind="line", marker="o", figsize=(7, 4))
plt.title("France et Allemagne, mois par mois")
plt.ylabel("CA (euros)")
plt.show()

mois_fr = par_mois["France"].idxmax()
print(mois_fr)

In [ ]:
verifier("1c - meilleur mois francais", mois_fr == "2011-10",
         "les deux pays sont France et Allemagne")

## 2. Les barres — comparer

**Toujours trier avant de tracer.** Un diagramme en barres non trié est
illisible.

In [ ]:
# nlargest(8) : le top 8. sort_values() : matplotlib dessine de bas en haut
ca_pays = complet.groupby("pays")["ca"].sum().nlargest(8).sort_values()

ca_pays.plot(kind="barh", figsize=(7, 4))   ## barh : les noms se lisent
plt.title("Chiffre d'affaires par pays (top 8)")
plt.xlabel("CA (euros)")
plt.show()

> 💡 **`barh` plutôt que `bar`.** En barres horizontales, les noms se lisent
> sans se chevaucher et sans rotation. Sur un écran étroit, c'est décisif.
>
> Et `sort_values()` **sans** `ascending=False` : matplotlib dessine de bas en
> haut, donc trier en ordre croissant met le plus grand tout en haut.

---

### ✏️ À vous 2a — Le rythme de la semaine

> **Votre mission :**
> - Calculer le CA par **jour de la semaine** dans `ca_jour`, trié du plus petit au plus grand.
> - Le tracer en barres horizontales, avec titre et unité.
> - Mettre le jour le plus fort dans `jour_top`.

In [ ]:
ca_jour = complet.groupby(complet["date"].dt.day_name())["ca"].sum().____()

ca_jour.plot(kind="____", figsize=(7, 4))
plt.title("Chiffre d'affaires par jour de la semaine")
plt.xlabel("CA (euros)")
plt.show()

jour_top = ca_jour.index[-1]
print(jour_top)

In [ ]:
verifier("2a - jour le plus fort", jour_top == "Thursday",
         "sort_values() trie ; apres un tri croissant le plus grand est en position -1")

---

### ✏️ À vous 2b — Compter n'est pas sommer

> **Votre mission :**
> - Combien de **références différentes** chaque catégorie contient-elle ? → `nb_ref`, trié, en barres horizontales.
> - Mettre la catégorie la plus fournie dans `cat_ref`.
> - Attention : on compte des produits, on n'additionne pas des euros. Ce n'est pas le même graphique ni la même conclusion.

In [ ]:
nb_ref = produits.groupby("categorie").____().sort_values()

nb_ref.plot(kind="barh", figsize=(7, 4))
plt.title("Nombre de references par categorie")
plt.xlabel("references")
plt.show()

cat_ref = nb_ref.index[-1]
print(cat_ref)

In [ ]:
verifier("2b - categorie la plus fournie", cat_ref == "deco",
         "size() compte les lignes, sum() additionnerait des valeurs")

## 3. L'histogramme — la répartition

In [ ]:
ventes["prix"].plot(kind="hist", bins=50, figsize=(7, 4))   ## 50 classes
plt.title("Repartition des prix unitaires")
plt.xlabel("Prix (euros)")
plt.show()

Illisible : une seule barre collée à gauche. En cause, le prix maximum à
4 161 €, qui étire tout l'axe.

**C'est une information, pas un problème.** Elle confirme ce qu'on avait vu
en séance 2.1 (moyenne 3,93 € contre médiane 1,95 €). Zoomons sur la zone
utile :

In [ ]:
# 85,8 % des ventes sont a moins de 5 euros
ventes.query("prix < 10")["prix"].plot(kind="hist", bins=40, figsize=(7, 4))

# Le zoom se dit DANS le titre : sinon on cache une information au lecteur
plt.title("Repartition des prix unitaires (moins de 10 euros)")
plt.xlabel("Prix (euros)")
plt.show()

Voilà l'entreprise réelle : **un vendeur de petits articles à moins de 5 €**,
en gros volumes. Ce n'est pas ce qu'une moyenne de 3,93 € laissait deviner —
elle aurait pu décrire aussi bien un catalogue homogène autour de 4 €.

> Quand vous zoomez pour rendre un graphique lisible, **dites-le dans le
> titre**. « moins de 10 euros » dans le titre ci-dessus : sans cette
> mention, vous cachez une information à votre lecteur.

---

### ✏️ À vous 3a — La répartition des quantités

> **Votre mission :**
> - Même technique, autre colonne : tracer l'histogramme des **quantités** commandées, en 50 classes.
> - Puis compter les lignes de plus de 100 unités → `nb_grosses`.

In [ ]:
complet["qte"].plot(kind="____", bins=50, figsize=(7, 4))
plt.title("Repartition des quantites commandees")
plt.xlabel("Quantite")
plt.show()

nb_grosses = len(complet.query("qte > ____"))
print(nb_grosses, "lignes de plus de 100 unites")

In [ ]:
verifier("3a - lignes de plus de 100 unites", nb_grosses == 560,
         "le type de graphique est hist ; le seuil de la question est 100")

---

### ✏️ À vous 3b — Zoomer, et le dire

> **Votre mission :**
> - L'histogramme précédent est illisible : la commande à 1 440 unités étire tout l'axe.
> - Le retracer sur les seules lignes de **moins de 25 unités**, et compter combien de lignes cela représente → `nb_petites`.
> - ⚠️ Le zoom doit apparaître **dans le titre** : sans ça, vous cachez une information à votre lecteur.
> - Quelle part du fichier ces lignes représentent-elles ?

In [ ]:
verifier("3b - lignes de moins de 25 unites", nb_petites == 41033,
         "filtrez avec query avant de tracer, puis len() sur le resultat")

## 4. Le nuage de points — chercher un lien

In [ ]:
# random_state=42 : le meme echantillon a chaque execution
echantillon = ventes.query("prix < 20 and qte < 200").sample(2000, random_state=42)

# alpha=0.3 : des points translucides, sinon 2 000 points font une tache
echantillon.plot(kind="scatter", x="prix", y="qte", alpha=0.3, figsize=(7, 4))
plt.title("Quantite commandee selon le prix unitaire")
plt.xlabel("Prix unitaire (euros)")
plt.ylabel("Quantite")
plt.show()

La relation est nette : **plus le prix unitaire est élevé, plus les quantités
commandées sont faibles.** Attendu, mais utile à vérifier.

> `alpha=0.3` rend les points semi-transparents : là où ils se superposent,
> la couleur devient plus dense. Sans ça, 2 000 points forment une bouillie
> noire.

> ⚠️ **Une relation n'est pas une cause.** Le prix ne « fait » pas baisser
> les quantités : ce sont deux conséquences du type de produit. Nous
> reviendrons sur cette distinction au bloc 5 (A/B testing) — c'est tout
> l'objet de l'expérimentation.

Ce nuage-ci ne montre que 2 000 lignes filtrées. À vous de le refaire sur le
fichier entier, et surtout de **mesurer** ce que l'œil croit voir.

---

### ✏️ À vous 4a — Mesurer le lien entre quantité et prix

> **Votre mission :**
> - Tracer un **nuage de points** avec `qte` en abscisse et `prix` en ordonnée.
> - `alpha=0.2` rend les points translucides : sans lui, 45 000 points forment une tache noire.
> - Mettre la corrélation entre les deux dans `lien`, arrondie à 3 décimales.

In [ ]:
complet.plot(kind="____", x="qte", y="prix", alpha=0.2, figsize=(7, 4))
plt.title("Quantite commandee et prix unitaire")
plt.show()

lien = round(complet["qte"].corr(complet["prix"]), 3)   ## entre -1 et +1
print(lien)

In [ ]:
verifier("4a - lien quantite / prix", lien == -0.024,
         "le type de graphique est scatter ; corr() donne le coefficient")

---

### ✏️ À vous 4b — Choisir le bon graphique

> **Votre mission :**
> - À chaque question sa figure. Compléter la liste `reponses` avec les quatre types, **dans l'ordre des questions** :
> - 1. Comment le chiffre d'affaires évolue-t-il dans le temps ?
> - 2. Quel pays est le plus gros marché ?
> - 3. Comment les prix sont-ils répartis ?
> - 4. Les grosses quantités vont-elles avec les prix bas ?

In [ ]:
reponses = ["____", "____", "____", "____"]

print(reponses)

In [ ]:
verifier("4b - le bon graphique pour la bonne question",
         reponses == ["line", "barh", "hist", "scatter"],
         "une evolution, un classement, une repartition, une relation")

## 5. Un graphique qu'on peut envoyer à un dirigeant

Les quatre lignes qui séparent un brouillon d'un livrable :

In [ ]:
ca_cat = complet.groupby("categorie")["ca"].sum().sort_values()

ca_cat.plot(kind="barh", figsize=(7, 4), color="#4C72B0")
plt.title("Chiffre d'affaires par categorie de produit, 2011")   ## quoi + quand
plt.xlabel("Chiffre d'affaires (euros)")   ## la grandeur et son unite
plt.ylabel("")            ## "categorie" n'apprend rien : on l'enleve
plt.tight_layout()        ## rien ne sera coupe au bord de la figure
plt.show()

- **`title`** : ce que montre le graphique, **et sur quelle période**
- **`xlabel` / `ylabel`** : le nom de la grandeur **et son unité**
- **`ylabel("")`** : on enlève l'étiquette « categorie », qui est évidente
- **`tight_layout()`** : évite que les étiquettes soient coupées

Si votre lecteur doit vous demander « c'est en quoi ? » ou « sur quelle
période ? », le graphique a échoué.

---

## Et maintenant : l'étude de cas

Vous avez tous les outils. Passez au notebook **« Étude de cas »**.

> 👥 **En binôme.** Un tient le clavier, l'autre lit l'énoncé et vérifie —
> puis vous échangez à mi-parcours. C'est la façon dont on travaille
> réellement sur une analyse, et c'est plus efficace que chacun de son côté.

---

## Ce que vous savez faire maintenant

| Votre question | Le graphique | La commande |
|---|---|---|
| comment ça évolue ? | courbe | `serie.plot(kind="line")` |
| qui est le plus gros ? | barres | `serie.plot(kind="barh")` |
| comment c'est réparti ? | histogramme | `df["prix"].plot(kind="hist", bins=30)` |
| y a-t-il un lien ? | nuage de points | `df.plot(kind="scatter", x="qte", y="prix")` |

Et toujours :

```python
plt.title("Ce que montre le graphique")
plt.xlabel("Nom de l'axe (unite)")
plt.ylabel("Nom de l'axe (unite)")
plt.show()
```

## Les trois réflexes qui font la différence

1. **Un graphique sans titre ni unité n'est pas un livrable.** Si votre
   lecteur doit vous demander « c'est en quoi ? », vous avez raté.
2. **Regardez toujours ce que le graphique ne montre pas.** Un mois incomplet,
   une catégorie absente, un axe qui ne part pas de zéro.
3. **Un chiffre ne devient une recommandation que quand il est rattaché à une
   décision.** « L'Irlande fait 22,7 % du CA » est un constat. « 22,7 % du CA
   repose sur deux comptes, il faut sécuriser ces contrats » est une
   recommandation.